# GPTNeo Training on TinyStories - A100 Optimized

Train a GPTNeo decoder-only transformer on TinyStories dataset with A100 GPU optimization.

**Features:**
- 🚀 BFloat16 mixed precision for 2x speedup
- 📚 100K TinyStories samples
- 🎯 ~85-95M parameter model
- ⚡ ~3-4 hours training time
- 📊 Expected PPL: 18-25

**Setup:**
1. Runtime → Change runtime type → A100 GPU
2. Run all cells
3. Training starts automatically

**Paper:** Eldan, R., & Li, Y. (2023). TinyStories: How Small Can Language Models Be and Still Speak Coherent English? arXiv:2305.07759.

## 1. Setup & Installation

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BFloat16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Install dependencies
!pip install -q transformers datasets tqdm tensorboard

print("✓ Dependencies installed")

In [ ]:
# Clone repository (if needed)
import os
if not os.path.exists('AttentionHeads'):
    !git clone https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")

# Change to mha directory
%cd AttentionHeads/mha

## 2. Configuration

In [ ]:
# Training configuration
config = {
    "model_name": "gptneo_tinystories_a100",
    "architecture": "gpt_neo_decoder",
    
    "model": {
        "vocab_size": 50257,
        "hidden_size": 768,
        "num_layers": 8,
        "num_heads": 12,
        "intermediate_size": 3072,
        "max_position_embeddings": 512,
        "dropout": 0.2,
        "activation": "gelu",
        "layer_norm_epsilon": 1e-5,
        "initializer_range": 0.02
    },
    
    "training": {
        "dataset": "tinystories",
        "train_samples": 100000,
        "val_samples": 5000,
        "batch_size": 128,
        "gradient_accumulation_steps": 1,
        "effective_batch_size": 128,
        "max_steps": 25000,
        "warmup_steps": 500,
        "learning_rate": 0.001,
        "min_learning_rate": 1e-5,
        "lr_scheduler": "cosine",
        "weight_decay": 0.01,
        "gradient_clip": 1.0,
        "use_bf16": True,
        "optimizer": "adamw",
        "adam_beta1": 0.9,
        "adam_beta2": 0.999,
        "adam_epsilon": 1e-8,
        "max_seq_length": 512
    },
    
    "data": {
        "tokenizer": "gpt2",
        "dataset_name": "roneneldan/TinyStories",
        "dataset_config": "default",
        "text_column": "text",
        "streaming": False,
        "num_workers": 4,
        "prefetch_factor": 2,
        "pin_memory": True
    },
    
    "logging": {
        "log_dir": "/content/logs/gptneo_tinystories",
        "checkpoint_dir": "/content/checkpoints/gptneo_tinystories",
        "save_every_steps": 1000,
        "eval_every_steps": 500,
        "log_every_steps": 50,
        "use_tensorboard": True,
        "use_wandb": False
    },
    
    "evaluation": {
        "eval_batch_size": 64,
        "eval_max_steps": 100,
        "generation_prompts": [
            "Once upon a time",
            "One day, a little girl",
            "In a big forest",
            "There was a happy dog"
        ],
        "generation_max_length": 100,
        "generation_temperature": 0.8,
        "generation_top_k": 50,
        "generation_top_p": 0.95
    },
    
    "checkpointing": {
        "save_total_limit": 3,
        "save_best_only": False
    },
    
    "random_seed": 42
}

# Save config
import json
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Configuration saved")
print(f"\nModel: {config['model']['hidden_size']}d, {config['model']['num_layers']} layers")
print(f"Training: {config['training']['train_samples']:,} samples, {config['training']['max_steps']:,} steps")
print(f"Batch size: {config['training']['batch_size']}")
print(f"Learning rate: {config['training']['learning_rate']}")
print(f"BFloat16: {config['training']['use_bf16']}")

## 3. Model Architecture

In [ ]:
# Import model classes
from transformer import GPTNeoForCausalLM, create_gptneo_model

# Create model
model = create_gptneo_model(config['model'])

# Model info
total_params = model.get_num_params()
non_embed_params = model.get_num_params(non_embedding=True)
embed_params = total_params - non_embed_params

print("GPTNeo Model Created")
print("=" * 50)
print(f"Total parameters: {total_params:,}")
print(f"Non-embedding parameters: {non_embed_params:,}")
print(f"Embedding parameters: {embed_params:,}")
print("=" * 50)

# Test forward pass
test_input = torch.randint(0, config['model']['vocab_size'], (2, 128))
test_output = model(test_input)
print(f"\nTest forward pass:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print("✓ Model working correctly")

## 4. Data Loading

In [ ]:
# Import data loader
from data_loader import TinyStoriesDataModule

# Create data module
data_cfg = {
    'dataset_name': config['data']['dataset_name'],
    'tokenizer': config['data']['tokenizer'],
    'train_samples': config['training']['train_samples'],
    'val_samples': config['training']['val_samples'],
    'batch_size': config['training']['batch_size'],
    'max_seq_length': config['training']['max_seq_length'],
    'num_workers': config['data']['num_workers'],
    'pin_memory': config['data']['pin_memory']
}

data_module = TinyStoriesDataModule(data_cfg)
data_module.setup()

# Test data loading
train_loader = data_module.train_dataloader()
batch = next(iter(train_loader))

print("\nData Loading Test:")
print(f"  Batch shape: {batch['input_ids'].shape}")
print(f"  Attention mask shape: {batch['attention_mask'].shape}")

# Decode a sample story
sample_ids = batch['input_ids'][0]
mask = batch['attention_mask'][0]
sample_ids = sample_ids[mask == 1]
sample_story = data_module.tokenizer.decode(sample_ids)

print(f"\nSample Story ({len(sample_ids)} tokens):")
print("-" * 50)
print(sample_story[:300] + "...")
print("-" * 50)
print("✓ Data loading working correctly")

## 5. Training Setup

In [ ]:
# Import trainer
from train import GPTNeoTrainer

# Create trainer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = GPTNeoTrainer(config, device=device)

print("✓ Trainer initialized")
print(f"\nTraining Configuration:")
print(f"  Device: {device}")
print(f"  Mixed precision: {'BFloat16' if trainer.use_bf16 else 'FP32'}")
print(f"  Steps: {config['training']['max_steps']:,}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Estimated time: ~3-4 hours on A100")

## 6. Start Training

**⚠️ This cell will run for ~3-4 hours on A100.**

Training will:
- Log metrics every 50 steps
- Evaluate every 500 steps
- Generate sample stories during evaluation
- Save checkpoints every 1000 steps
- Save best model based on validation loss

In [ ]:
# Start training
trainer.train()

## 7. Evaluation & Generation

In [ ]:
# Load best model
import torch
from transformer import GPTNeoForCausalLM
from transformers import GPT2Tokenizer

# Load checkpoint
checkpoint_path = '/content/checkpoints/gptneo_tinystories/best_model.pt'
checkpoint = torch.load(checkpoint_path)

# Create model and load weights
model = create_gptneo_model(config['model'])
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

print("✓ Best model loaded")
print(f"Training step: {checkpoint['global_step']:,}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")
print(f"Validation PPL: {checkpoint.get('metrics', {}).get('val_ppl', 'N/A')}")

In [ ]:
# Generate stories
def generate_story(prompt, max_length=200, temperature=0.8, top_k=50, top_p=0.95):
    """Generate a story from a prompt"""
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )
    
    story = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return story

# Test prompts
prompts = [
    "Once upon a time",
    "One day, a little girl",
    "In a big forest",
    "There was a happy dog",
    "A small boy found"
]

print("\nGenerated Stories:")
print("=" * 80)

for i, prompt in enumerate(prompts, 1):
    story = generate_story(prompt, max_length=150, temperature=0.8)
    print(f"\n{i}. Prompt: \"{prompt}\"")
    print("-" * 80)
    print(story)
    print("-" * 80)

In [ ]:
# Interactive generation
print("\nInteractive Story Generation")
print("=" * 80)
print("Enter your own prompts to generate stories!\n")

while True:
    prompt = input("Enter prompt (or 'quit' to exit): ")
    if prompt.lower() in ['quit', 'exit', 'q']:
        break
    
    if not prompt.strip():
        continue
    
    story = generate_story(prompt, max_length=200, temperature=0.8)
    print("\nGenerated Story:")
    print("-" * 80)
    print(story)
    print("-" * 80 + "\n")

## 8. Analyze Results

In [ ]:
# Plot training curves (if tensorboard logs available)
import matplotlib.pyplot as plt
import numpy as np

# Load TensorBoard logs
from tensorboard.backend.event_processing import event_accumulator

log_dir = '/content/logs/gptneo_tinystories'

try:
    ea = event_accumulator.EventAccumulator(log_dir)
    ea.Reload()
    
    # Get metrics
    train_loss = ea.Scalars('train/loss')
    val_loss = ea.Scalars('val/loss')
    train_ppl = ea.Scalars('train/perplexity')
    val_ppl = ea.Scalars('val/perplexity')
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    axes[0].plot([x.step for x in train_loss], [x.value for x in train_loss], label='Train')
    axes[0].plot([x.step for x in val_loss], [x.value for x in val_loss], label='Val', marker='o')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Perplexity
    axes[1].plot([x.step for x in train_ppl], [x.value for x in train_ppl], label='Train')
    axes[1].plot([x.step for x in val_ppl], [x.value for x in val_ppl], label='Val', marker='o')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Perplexity')
    axes[1].set_title('Training & Validation Perplexity')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Training curves saved to /content/training_curves.png")
    
except Exception as e:
    print(f"Could not load TensorBoard logs: {e}")
    print("Run TensorBoard manually to view metrics")

## 9. Save Model to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create save directory
save_dir = '/content/drive/MyDrive/GPTNeo_TinyStories'
!mkdir -p {save_dir}

# Copy checkpoints
!cp -r /content/checkpoints/gptneo_tinystories {save_dir}/
!cp /content/training_curves.png {save_dir}/
!cp config.json {save_dir}/

print(f"✓ Model saved to Google Drive: {save_dir}")
print("\nFiles saved:")
!ls -lh {save_dir}

## 10. Summary

### Training Complete! 🎉

**What You Trained:**
- Model: GPTNeo decoder-only (~85-95M parameters)
- Dataset: TinyStories (100K samples)
- Training: 25K steps with BFloat16 on A100

**Expected Results:**
- Validation PPL: 18-25
- Training time: ~3-4 hours
- Story quality: Coherent children's stories

**Next Steps:**
1. Try different prompts in the interactive generator
2. Fine-tune hyperparameters in config
3. Train longer (50K steps) for better results
4. Compare with other attention mechanisms

**Citation:**
```
Eldan, R., & Li, Y. (2023). TinyStories: How Small Can Language Models Be 
and Still Speak Coherent English? arXiv:2305.07759.
```